# Praktikum 4: Perbandingan Performa Annoy, FAISS, dan HNSW

In [1]:
# Instalasi library jika belum terpasang
!pip install annoy faiss-cpu hnswlib

import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 14.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.7 MB/s eta 0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520034 sha256=8e967d4bde8ec8c4a5830d3fb1603785e4cd7ede621f588bcf27d62defd56542
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679797 sha256=20a82f242f3977492bbeb91e64720ada3d9cea15f41d731924a04d426e66137a
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built annoy hnswlib


In [2]:
# ===============================
# 1. Buat dataset 1 juta data 5D
# ===============================
n_data = 1_000_000   # bisa disesuaikan jika RAM terbatas
dim = 5
np.random.seed(42)
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time_annoy = time.time() - start

start = time.time()
neighbors_annoy = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time_annoy = time.time() - start

print("Build time:", round(build_time_annoy, 4), "detik")
print("Query time:", round(query_time_annoy, 6), "detik")
print("Neighbors:", neighbors_annoy[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time_faiss = time.time() - start

start = time.time()
distances_faiss, indices_faiss = faiss_index.search(query, k)
query_time_faiss = time.time() - start

print("Build time:", round(build_time_faiss, 4), "detik")
print("Query time:", round(query_time_faiss, 6), "detik")
print("Neighbors:", indices_faiss[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time_hnsw = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels_hnsw, distances_hnsw = hnsw_index.knn_query(query, k=k)
query_time_hnsw = time.time() - start

print("Build time:", round(build_time_hnsw, 4), "detik")
print("Query time:", round(query_time_hnsw, 6), "detik")
print("Neighbors:", labels_hnsw[0][:5], "...")

=== Annoy ===
Build time: 24.1293 detik
Query time: 0.000299 detik
Neighbors: [901095, 495168, 561017, 163986, 185734] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.016 detik
Query time: 0.010026 detik
Neighbors: [901095 495168 561017 163986 185734] ...

=== HNSW (hnswlib) ===
Build time: 184.9957 detik
Query time: 0.00024 detik
Neighbors: [901095 495168 561017 163986 185734] ...


In [3]:
def run_comparison_experiment(metric_name, n_data=1_000_000, dim=5, k=10):
    np.random.seed(42)
    X_exp = np.random.random((n_data, dim)).astype(np.float32)
    query_exp = np.random.random((1, dim)).astype(np.float32)

    annoy_metric = 'euclidean' if metric_name == 'Euclidean' else 'angular'
    hnsw_space = 'l2' if metric_name == 'Euclidean' else 'cosine'

    print(f"\n================ METRIC: {metric_name} ================")

    # 1. Annoy
    t_ann = AnnoyIndex(dim, annoy_metric)
    t0 = time.time()
    for i in range(n_data):
        t_ann.add_item(i, X_exp[i])
    t_ann.build(10)
    ann_btime = time.time() - t0

    t0 = time.time()
    ann_res = t_ann.get_nns_by_vector(query_exp[0], k)
    ann_qtime = time.time() - t0

    # 2. FAISS
    if metric_name == 'Euclidean':
        f_idx = faiss.IndexFlatL2(dim)
        X_faiss = X_exp
        q_faiss = query_exp
    else:
        f_idx = faiss.IndexFlatIP(dim)
        X_faiss = X_exp.copy()
        q_faiss = query_exp.copy()
        faiss.normalize_L2(X_faiss)
        faiss.normalize_L2(q_faiss)

    t0 = time.time()
    f_idx.add(X_faiss)
    faiss_btime = time.time() - t0

    t0 = time.time()
    _, faiss_res = f_idx.search(q_faiss, k)
    faiss_qtime = time.time() - t0

    # 3. HNSW
    h_idx = hnswlib.Index(space=hnsw_space, dim=dim)
    t0 = time.time()
    h_idx.init_index(max_elements=n_data, ef_construction=200, M=16)
    h_idx.add_items(X_exp)
    hnsw_btime = time.time() - t0

    h_idx.set_ef(50)
    t0 = time.time()
    hnsw_res, _ = h_idx.knn_query(query_exp, k=k)
    hnsw_qtime = time.time() - t0

    print(f"Annoy  | Build: {ann_btime:.4f} s | Query: {ann_qtime*1000:.4f} ms | Top 5: {ann_res[:5]}")
    print(f"FAISS  | Build: {faiss_btime:.4f} s | Query: {faiss_qtime*1000:.4f} ms | Top 5: {list(faiss_res[0][:5])}")
    print(f"HNSW   | Build: {hnsw_btime:.4f} s | Query: {hnsw_qtime*1000:.4f} ms | Top 5: {list(hnsw_res[0][:5])}")

run_comparison_experiment('Euclidean')
run_comparison_experiment('Cosine/Angular')


================ METRIC: Euclidean ================
Annoy  | Build: 26.4415 s | Query: 0.1314 ms | Top 5: [901095, 495168, 561017, 163986, 185734]
FAISS  | Build: 0.0220 s | Query: 9.8462 ms | Top 5: [np.int64(901095), np.int64(495168), np.int64(561017), np.int64(163986), np.int64(185734)]
HNSW   | Build: 180.0014 s | Query: 0.1431 ms | Top 5: [np.uint64(901095), np.uint64(495168), np.uint64(561017), np.uint64(163986), np.uint64(185734)]

================ METRIC: Cosine/Angular ================
Annoy  | Build: 30.6148 s | Query: 0.1597 ms | Top 5: [653774, 135588, 561017, 901095, 211473]
FAISS  | Build: 0.0351 s | Query: 4.6976 ms | Top 5: [np.int64(653774), np.int64(135588), np.int64(561017), np.int64(901095), np.int64(211473)]
HNSW   | Build: 193.8131 s | Query: 0.2069 ms | Top 5: [np.uint64(653774), np.uint64(135588), np.uint64(561017), np.uint64(901095), np.uint64(211473)]
